In [1]:
from mibici.pipeline import load_stage
import pandas as pd

data = load_stage("raw")

trips = data["viajes"]
stations = data["estaciones"]

print(f"Viajes: {trips.shape}")
print(f"Estaciones: {stations.shape}")

2026-10-18 11:56:55.367 | INFO     | mibici.config:<module>:11 - PROJ_ROOT path is: /home/jovyan/notebooks/mibici/mibici_predictions


2026-10-18 11:56:57.695 | INFO     | mibici.dataset:load_raw:124 - Leyendo archivo: datos_abiertos_2024_10.csv
2026-10-18 11:56:58.151 | INFO     | mibici.dataset:load_raw:124 - Leyendo archivo: datos_abiertos_2024_11.csv
2026-10-18 11:56:58.700 | INFO     | mibici.dataset:load_raw:124 - Leyendo archivo: datos_abiertos_2024_12.csv
2026-10-18 11:56:59.196 | INFO     | mibici.dataset:load_raw:124 - Leyendo archivo: datos_abiertos_2025_01.csv
2026-10-18 11:56:59.734 | INFO     | mibici.dataset:load_raw:124 - Leyendo archivo: datos_abiertos_2025_02.csv
2026-10-18 11:57:00.285 | INFO     | mibici.dataset:load_raw:124 - Leyendo archivo: datos_abiertos_2025_03.csv
2026-10-18 11:57:00.856 | INFO     | mibici.dataset:load_raw:124 - Leyendo archivo: datos_abiertos_2025_04.csv
2026-10-18 11:57:01.396 | INFO     | mibici.dataset:load_raw:124 - Leyendo archivo: datos_abiertos_2025_05.csv
2026-10-18 11:57:01.940 | INFO     | mibici.dataset:load_raw:124 - Leyendo archivo: datos_abiertos_2025_06.csv
2

In [2]:
display(trips.head())
display(stations.head())

,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id
0,35847248,701161,M,1983.0,01/10/2024 0:00,01/10/2024 0:21,40,213
1,35847249,4453122,M,1997.0,01/10/2024 0:01,01/10/2024 0:18,189,3
2,35847250,302530,M,1994.0,01/10/2024 0:01,01/10/2024 0:11,131,342
3,35847251,4062235,F,1999.0,01/10/2024 0:01,01/10/2024 0:18,189,3
4,35847252,2664376,M,1979.0,01/10/2024 0:02,01/10/2024 0:12,244,46


,id,name,latitude,longitude,dpcapacity
0,2,(GDL-001) C. Epigmenio Glez./ Av. 16 de Sept.,20.666378,-103.348820,15
1,3,(GDL-002) C. Colonias / Av. Niños héroes,20.667228,-103.366000,15
2,4,(GDL-003) C. Vidrio / Av. Chapultepec,20.667690,-103.368252,19
3,5,(GDL-004) C. Ghilardi /C. Miraflores,20.691847,-103.362549,19
4,6,(GDL-005) C. San Diego /Calzada Independencia,20.681158,-103.339363,11


In [3]:
expected_trip_types = {
    "Viaje_Id": "Int64",
    "Usuario_Id": "Int64",
    "Genero": "string",
    "Año_de_nacimiento": "Int64",
    "Inicio_del_viaje": "datetime64[ns]",
    "Fin_del_viaje": "datetime64[ns]",
    "Origen_Id": "Int64",
    "Destino_Id": "Int64",
}

expected_station_types = {
    "id": "Int64",
    "name": "string",
    "latitude": "float64",
    "longitude": "float64",
    "dpcapacity": "Int64",
}

def compare_types(df, expected_types):
    return pd.DataFrame({
        "loaded_type": df.dtypes.astype(str),
        "expected_type": pd.Series(expected_types),
    })

In [4]:
# Comparando los tipos de datos de los viajes y las estaciones con los tipos esperados
display(compare_types(trips, expected_trip_types))

,loaded_type,expected_type
Viaje_Id,int64,Int64
Usuario_Id,int64,Int64
Genero,object,string
Año_de_nacimiento,float64,Int64
Inicio_del_viaje,object,datetime64[ns]
Fin_del_viaje,object,datetime64[ns]
Origen_Id,int64,Int64
Destino_Id,int64,Int64


In [5]:
# Revisando porque algunos valores de Año_de_nacimiento tienen float
birth_year = trips["Año_de_nacimiento"]

print(f"Valores faltantes: {birth_year.isna().sum():,}")

fractional_years = birth_year[
    birth_year.notna() & birth_year.mod(1).ne(0)
]

print(f"Valores con parte decimal: {len(fractional_years):,}")
display(fractional_years.head())

Valores faltantes: 60
Valores con parte decimal: 0


Series([], Name: Año_de_nacimiento, dtype: float64)

In [6]:
birth_year = trips["Año_de_nacimiento"].astype("Int64")

print(f"Tipo: {birth_year.dtype}")
print(f"Faltantes: {birth_year.isna().sum()}")
display(birth_year.head())

Tipo: Int64
Faltantes: 60


0    1968
1    1997
2    2002
3    1992
4    2006
Name: Año_de_nacimiento, dtype: Int64

In [ ]:
# Analizando la columna de género

gender_counts = trips["Genero"].value_counts(dropna=False)
display(gender_counts)

Genero
M      636737
F      241216
NaN      5627
Name: count, dtype: int64

In [ ]:
#Verificando el tipo de datos de la columna de género
gender = trips["Genero"].astype("string")

print(f"Tipo: {gender.dtype}")
print(f"Faltantes: {gender.isna().sum():,}")
display(gender.value_counts(dropna=False))

Tipo: string
Faltantes: 5,627


Genero
M       636737
F       241216
<NA>      5627
Name: count, dtype: Int64

In [ ]:
date_columns = ["Inicio_del_viaje", "Fin_del_viaje"]

for column in date_columns:
    original = trips[column].astype("string")
    slash_dates = original.str.contains("/", regex=False, na=False)

    parsed = pd.Series(
        pd.NaT,
        index=trips.index,
        dtype="datetime64[ns]",
    )

    parsed.loc[slash_dates] = pd.to_datetime(
        original.loc[slash_dates],
        format="%d/%m/%Y %H:%M",
        errors="coerce",
    )

    parsed.loc[~slash_dates] = pd.to_datetime(
        original.loc[~slash_dates],
        format="%Y-%m-%d %H:%M:%S",
        errors="coerce",
    )

    invalid_dates = original.notna() & parsed.isna()

    print(f"\nColumna: {column}")
    print(f"Tipo convertido: {parsed.dtype}")
    print(f"Faltantes originales: {original.isna().sum():,}")
    print(f"Valores no interpretables: {invalid_dates.sum():,}")

    display(original.loc[invalid_dates].head())


Columna: Inicio_del_viaje
Tipo convertido: datetime64[ns]
Faltantes originales: 0
Valores que no pudieron convertirse: 0


Series([], Name: Inicio_del_viaje, dtype: object)


Columna: Fin_del_viaje
Tipo convertido: datetime64[ns]
Faltantes originales: 0
Valores que no pudieron convertirse: 0


Series([], Name: Fin_del_viaje, dtype: object)

In [10]:
display(compare_types(stations, expected_station_types))

,loaded_type,expected_type
id,int64,Int64
name,object,string
latitude,float64,float64
longitude,float64,float64
dpcapacity,int64,Int64


In [11]:
# Anlizando la columna name de las estaciones

station_names = stations["name"].astype("string")

print(f"Tipo: {station_names.dtype}")
print(f"Faltantes originales: {stations['name'].isna().sum():,}")
print(f"Faltantes después: {station_names.isna().sum():,}")
display(station_names.head())

Tipo: string
Faltantes originales: 0
Faltantes después: 0


0    (GDL-001) C. Epigmenio Glez./ Av. 16 de Sept.
1       (GDL-002) C. Colonias  / Av.  Niños héroes
2            (GDL-003) C. Vidrio / Av. Chapultepec
3             (GDL-004) C. Ghilardi /C. Miraflores
4    (GDL-005) C. San Diego /Calzada Independencia
Name: name, dtype: string

### Conclusiones

Se revisaron los tipos de datos de viajes de agosto y septiembre
de 2026 y del catálogo de estaciones de septiembre de 2026.

Conversiones propuestas:
- Año_de_nacimiento → Int64, conservando los faltantes.
- Genero y name → string.
- Inicio_del_viaje y Fin_del_viaje → datetime64[ns].

Los identificadores, coordenadas y capacidad conservan sus tipos actuales.
Las conversiones están pendientes de integrar al pipeline.